# Programación dinámica en árboles

En un árbol, cada subárbol es un subproblema independiente: si se cuelga el árbol de una raíz, la solución de un nodo $v$ se combina a partir de las de sus hijos, $D[v] = f\big(D[h_1], \dots, D[h_k]\big)$. Basta procesar los nodos en **postorden** (los hijos antes que el padre) para rellenar la tabla en $O(n)$, porque cada arista se mira una sola vez. Cuando hace falta la respuesta tomando cada nodo como raíz, el **rerooting** (cambio de raíz) hace una segunda pasada en preorden que pasa a cada hijo la información del resto del árbol, y sigue costando $O(n)$ en lugar de $O(n^2)$.

Los árboles se guardan como listas de adyacencia y se recorren sin recursión para no chocar con el límite de recursión de Python en árboles largos. Las soluciones se comprueban contra fuerza bruta en árboles aleatorios pequeños.

## 1. Colgar el árbol de una raíz

Escribir `arbol_aleatorio(n, semilla)`, que genere un árbol de $n$ nodos uniendo cada nodo $i > 0$ con uno anterior al azar, y `colgar(adj, raiz)`, que devuelva el padre de cada nodo y un orden en el que cada padre aparece antes que sus hijos (un BFS). Recorriendo ese orden al revés se obtiene un postorden válido: calcular así el tamaño de cada subárbol y comprobarlo contra un recorrido en profundidad desde cada nodo que no cruce hacia el padre.

In [1]:
import random
from collections import deque
from itertools import combinations, product


def arbol_aleatorio(n, semilla):
    rng = random.Random(semilla)
    adj = [[] for _ in range(n)]
    for i in range(1, n):
        p = rng.randrange(i)
        adj[i].append(p)
        adj[p].append(i)
    return adj


def colgar(adj, raiz=0):
    padre = [-1] * len(adj)
    orden = [raiz]
    visto = [False] * len(adj)
    visto[raiz] = True
    for v in orden:  # la lista crece mientras se recorre: es un BFS
        for w in adj[v]:
            if not visto[w]:
                visto[w] = True
                padre[w] = v
                orden.append(w)
    return padre, orden


def tamanos(adj, raiz=0):
    padre, orden = colgar(adj, raiz)
    tam = [1] * len(adj)
    for v in reversed(orden):
        if padre[v] >= 0:
            tam[padre[v]] += tam[v]
    return tam


def tamano_bruto(adj, v, padre):
    pila, cuenta = [(v, padre)], 0
    while pila:
        x, p = pila.pop()
        cuenta += 1
        pila.extend((w, x) for w in adj[x] if w != p)
    return cuenta


adj = arbol_aleatorio(10, 1)
padre, orden = colgar(adj)
print("Aristas:", [(padre[v], v) for v in orden[1:]])
print("Tamaños:", tamanos(adj))
for semilla in range(50):
    adj = arbol_aleatorio(random.Random(semilla).randint(1, 30), semilla)
    padre, _ = colgar(adj)
    assert tamanos(adj) == [tamano_bruto(adj, v, padre[v]) for v in range(len(adj))]

Aristas: [(0, 1), (0, 2), (0, 4), (1, 3), (3, 5), (3, 6), (3, 7), (3, 9), (6, 8)]
Tamaños: [10, 7, 1, 6, 1, 1, 2, 1, 1, 1]


## 2. Conjunto independiente de peso máximo

Las cofradías de un pueblo están unidas por una red de amistades en forma de árbol y cada una aporta $p_v$ voluntarios. Para la romería se quiere elegir cofradías que no sean vecinas en el árbol y sumar el máximo de voluntarios. Con $T[v]$ el mejor peso del subárbol de $v$ tomando $v$ y $N[v]$ sin tomarlo:

$$T[v] = p_v + \sum_h N[h], \qquad N[v] = \sum_h \max(T[h], N[h])$$

Reconstruir el conjunto bajando desde la raíz y comprobarlo contra todos los subconjuntos en árboles de hasta 12 nodos.

In [2]:
def independiente_maximo(adj, pesos, raiz=0):
    padre, orden = colgar(adj, raiz)
    T = list(pesos)
    N = [0] * len(adj)
    for v in reversed(orden):
        p = padre[v]
        if p >= 0:
            T[p] += N[v]
            N[p] += max(T[v], N[v])
    elegidos, tomado = [], [False] * len(adj)
    for v in orden:
        padre_tomado = padre[v] >= 0 and tomado[padre[v]]
        if not padre_tomado and T[v] >= N[v]:
            tomado[v] = True
            elegidos.append(v)
    return max(T[raiz], N[raiz]), sorted(elegidos)


def aristas(adj):
    return [(u, v) for u in range(len(adj)) for v in adj[u] if u < v]


def independiente_bruto(adj, pesos):
    mejor = 0
    for mascara in range(1 << len(adj)):
        if all(not (mascara >> u & 1 and mascara >> v & 1) for u, v in aristas(adj)):
            mejor = max(mejor, sum(p for i, p in enumerate(pesos) if mascara >> i & 1))
    return mejor


rng = random.Random(2)
adj = arbol_aleatorio(10, 3)
pesos = [rng.randint(1, 20) for _ in adj]
valor, conjunto = independiente_maximo(adj, pesos)
print("Aristas:", aristas(adj))
print("Pesos:  ", pesos)
print(f"Máximo: {valor} con las cofradías {conjunto}")
assert valor == independiente_bruto(adj, pesos)

for semilla in range(60):
    n = rng.randint(1, 12)
    adj = arbol_aleatorio(n, semilla)
    pesos = [rng.randint(0, 15) for _ in range(n)]
    valor, conjunto = independiente_maximo(adj, pesos)
    s = set(conjunto)
    assert all(not (u in s and v in s) for u, v in aristas(adj))
    assert sum(pesos[v] for v in conjunto) == valor == independiente_bruto(adj, pesos)

Aristas: [(0, 1), (0, 2), (0, 6), (0, 8), (1, 3), (3, 4), (4, 5), (4, 7), (7, 9)]
Pesos:   [2, 3, 3, 12, 6, 10, 9, 20, 7, 20]
Máximo: 61 con las cofradías [2, 3, 5, 6, 7, 8]


## 3. Cubrimiento mínimo de vértices

Se quieren poner cámaras en los cruces de una red de senderos con forma de árbol de modo que cada sendero tenga una cámara en al menos uno de sus dos extremos. Calcular el mínimo de cámaras: si $v$ no tiene cámara, todos sus hijos deben tenerla, así que $C[v] = 1 + \sum_h \min(C[h], S[h])$ y $S[v] = \sum_h C[h]$. Comprobarlo contra fuerza bruta y verificar que, en un árbol (que es bipartito), el mínimo cubrimiento más el máximo conjunto independiente (con pesos 1) suman $n$.

In [3]:
def cubrimiento_minimo(adj, raiz=0):
    padre, orden = colgar(adj, raiz)
    C = [1] * len(adj)
    S = [0] * len(adj)
    for v in reversed(orden):
        p = padre[v]
        if p >= 0:
            C[p] += min(C[v], S[v])
            S[p] += C[v]
    return min(C[raiz], S[raiz])


def cubrimiento_bruto(adj):
    n = len(adj)
    for k in range(n + 1):
        for camaras in combinations(range(n), k):
            s = set(camaras)
            if all(u in s or v in s for u, v in aristas(adj)):
                return k


adj = arbol_aleatorio(12, 7)
print("Aristas:", aristas(adj))
print("Cámaras necesarias:", cubrimiento_minimo(adj))
for semilla in range(60):
    n = random.Random(semilla).randint(1, 12)
    adj = arbol_aleatorio(n, semilla)
    c = cubrimiento_minimo(adj)
    assert c == cubrimiento_bruto(adj)
    assert c + independiente_maximo(adj, [1] * n)[0] == n

Aristas: [(0, 1), (0, 2), (0, 4), (0, 5), (0, 7), (0, 9), (1, 3), (3, 11), (4, 6), (5, 8), (8, 10)]
Cámaras necesarias: 4


## 4. La fiesta de la empresa

Una empresa de La Laguna organiza su cena de Navidad. El organigrama es un árbol con la presidenta en la raíz y cada persona tiene una puntuación de animación; nadie disfruta si va también su jefe directo. Elegir a los invitados que maximizan la animación total con estas dos condiciones extra:

- la presidenta tiene que ir (se fuerza $T$ en la raíz);
- el restaurante solo admite $k$ personas: añadir al estado el número de invitados y combinar los hijos como en una mochila, $D[v][j]$, lo que cuesta $O(n^2)$ en total porque cada par de nodos se combina una vez.

Comprobar las dos versiones contra fuerza bruta.

In [4]:
organigrama = {
    "Presidenta": ["Dir. técnica", "Dir. comercial", "Dir. financiero"],
    "Dir. técnica": ["Ana", "Borja", "Carmen"],
    "Dir. comercial": ["Dácil", "Eloy"],
    "Dir. financiero": ["Fayna"],
    "Ana": ["Gara", "Himar"],
    "Eloy": ["Iballa", "Jonay", "Kevin"],
}
animacion = {
    "Presidenta": 1, "Dir. técnica": 12, "Dir. comercial": 9, "Dir. financiero": 7, "Ana": 8, "Borja": 5,
    "Carmen": 6, "Dácil": 7, "Eloy": 10, "Fayna": 6, "Gara": 4, "Himar": 3, "Iballa": 5, "Jonay": 2, "Kevin": 4,
}
nombres = list(animacion)
indice = {nombre: i for i, nombre in enumerate(nombres)}
adj_emp = [[] for _ in nombres]
for jefe, equipo in organigrama.items():
    for persona in equipo:
        adj_emp[indice[jefe]].append(indice[persona])
        adj_emp[indice[persona]].append(indice[jefe])
puntos = [animacion[x] for x in nombres]

NEG = float("-inf")


def fiesta(adj, pesos, k, raiz=0):
    """D[v][t][j]: mejor animación del subárbol de v con j invitados; t = 1 si v va."""
    padre, orden = colgar(adj, raiz)
    D = [[[0, NEG], [NEG, p]] for p in pesos]
    for v in reversed(orden):
        p = padre[v]
        if p < 0:
            continue
        mejor_h = [max(a, b) for a, b in zip(*D[v])]  # el hijo va o no
        no_h = D[v][0]
        nuevo = []
        for t, tabla_hijo in ((0, mejor_h), (1, no_h)):
            fila = [NEG] * min(k + 1, len(D[p][t]) + len(tabla_hijo) - 1)
            for i, a in enumerate(D[p][t]):
                if a == NEG:
                    continue
                for j, b in enumerate(tabla_hijo):
                    if i + j < len(fila) and b != NEG:
                        fila[i + j] = max(fila[i + j], a + b)
            nuevo.append(fila)
        D[p] = nuevo
    return D[raiz]


def fiesta_bruta(adj, pesos, k, raiz=0):
    padre, _ = colgar(adj, raiz)
    mejor = {False: NEG, True: NEG}
    for mascara in range(1 << len(adj)):
        invitados = [v for v in range(len(adj)) if mascara >> v & 1]
        if len(invitados) <= k and all(padre[v] < 0 or not mascara >> padre[v] & 1 for v in invitados):
            va = bool(mascara >> raiz & 1)
            mejor[va] = max(mejor[va], sum(pesos[v] for v in invitados))
    return mejor


n = len(nombres)
D = fiesta(adj_emp, puntos, n)
sin_limite = max(max(D[0]), max(D[1]))
con_presidenta = max(D[1])
valor, invitados = independiente_maximo(adj_emp, puntos)
print(f"Sin condiciones: {sin_limite} con", ", ".join(nombres[v] for v in invitados))
print(f"Con la presidenta: {con_presidenta}")
print(f"{'k':>2} {'mejor':>6} {'con presidenta':>15}")
for k in range(1, 9):
    Dk = fiesta(adj_emp, puntos, k)
    print(f"{k:>2} {max(max(Dk[0]), max(Dk[1])):>6} {max(Dk[1]):>15}")
assert sin_limite == valor > con_presidenta

rng = random.Random(4)
for semilla in range(40):
    m = rng.randint(1, 11)
    adj = arbol_aleatorio(m, semilla)
    pesos = [rng.randint(1, 9) for _ in range(m)]
    k = rng.randint(1, m)
    D = fiesta(adj, pesos, k)
    bruto = fiesta_bruta(adj, pesos, k)
    assert max(D[1]) == bruto[True] and max(max(D[0]), max(D[1])) == max(bruto.values())

Sin condiciones: 46 con Dir. técnica, Dir. comercial, Dir. financiero, Gara, Himar, Iballa, Jonay, Kevin
Con la presidenta: 44
 k  mejor  con presidenta
 1     12               1
 2     22              11
 3     29              19
 4     36              26
 5     40              32
 6     43              38
 7     44              43
 8     46              43


## 5. Diámetro de un árbol

El diámetro es el camino más largo entre dos nodos. En postorden, guardar en $A[v]$ la profundidad máxima que se alcanza bajando desde $v$; el camino más largo cuyo punto más alto es $v$ suma las dos mayores $A[h] + w(v, h)$ de sus hijos. Funciona también con pesos no negativos en las aristas (distancias en km entre pueblos). Comparar con el truco de las dos búsquedas (desde cualquier nodo, ir al más lejano y, desde él, otra vez al más lejano) y con la distancia máxima entre todos los pares.

In [5]:
def colgar_pesado(adj_w, raiz=0):
    adj = [[w for w, _ in vecinos] for vecinos in adj_w]
    return colgar(adj, raiz)


def diametro(adj_w):
    padre, orden = colgar_pesado(adj_w)
    A = [0] * len(adj_w)
    mejor = 0
    for v in reversed(orden):
        primero = segundo = 0
        for h, w in adj_w[v]:
            if h != padre[v]:
                d = A[h] + w
                if d > primero:
                    primero, segundo = d, primero
                elif d > segundo:
                    segundo = d
        A[v] = primero
        mejor = max(mejor, primero + segundo)
    return mejor


def distancias_desde(adj_w, origen):
    dist = [None] * len(adj_w)
    dist[origen] = 0
    pila = [origen]
    while pila:
        v = pila.pop()
        for h, w in adj_w[v]:
            if dist[h] is None:
                dist[h] = dist[v] + w
                pila.append(h)
    return dist


def diametro_dos_busquedas(adj_w):
    d0 = distancias_desde(adj_w, 0)
    a = d0.index(max(d0))
    return max(distancias_desde(adj_w, a))


def con_pesos(adj, rng, maximo=9):
    adj_w = [[] for _ in adj]
    for u, v in aristas(adj):
        w = rng.randint(0, maximo)
        adj_w[u].append((v, w))
        adj_w[v].append((u, w))
    return adj_w


pueblos = ["Arucas", "Firgas", "Moya", "Teror", "Valleseco", "Gáldar", "Guía", "Agaete"]
carreteras = [(0, 1, 6), (1, 2, 7), (0, 3, 9), (3, 4, 8), (2, 6, 11), (6, 5, 3), (5, 7, 9)]
adj_w = [[] for _ in pueblos]
for u, v, km in carreteras:
    adj_w[u].append((v, km))
    adj_w[v].append((u, km))
print("Diámetro de la red:", diametro(adj_w), "km")
assert diametro(adj_w) == diametro_dos_busquedas(adj_w) == 8 + 9 + 6 + 7 + 11 + 3 + 9  # de Valleseco a Agaete

rng = random.Random(5)
for semilla in range(100):
    adj_w = con_pesos(arbol_aleatorio(rng.randint(1, 25), semilla), rng)
    todos = max(max(distancias_desde(adj_w, v)) for v in range(len(adj_w)))
    assert diametro(adj_w) == diametro_dos_busquedas(adj_w) == todos

Diámetro de la red: 53 km


## 6. Suma de distancias desde cada nodo (*rerooting*)

Para decidir dónde poner el almacén de una red de reparto con forma de árbol, calcular para cada nodo $v$ la suma $S[v]$ de las distancias (en aristas) a todos los demás. Una pasada en postorden da $S[\text{raíz}]$ y los tamaños; al mover la raíz de $p$ a su hijo $h$, los $t_h$ nodos del subárbol de $h$ quedan un paso más cerca y los $n - t_h$ restantes un paso más lejos:

$$S[h] = S[p] - t_h + (n - t_h)$$

Comprobarlo contra un BFS desde cada nodo y encontrar el mejor sitio para el almacén (la mediana del árbol).

In [6]:
def suma_distancias(adj):
    n = len(adj)
    padre, orden = colgar(adj)
    tam = [1] * n
    abajo = [0] * n  # suma de distancias de v a su subárbol
    for v in reversed(orden):
        p = padre[v]
        if p >= 0:
            tam[p] += tam[v]
            abajo[p] += abajo[v] + tam[v]
    S = [0] * n
    S[orden[0]] = abajo[orden[0]]
    for v in orden[1:]:
        S[v] = S[padre[v]] - tam[v] + (n - tam[v])
    return S


def bfs(adj, origen):
    dist = [-1] * len(adj)
    dist[origen] = 0
    cola = deque([origen])
    while cola:
        v = cola.popleft()
        for w in adj[v]:
            if dist[w] < 0:
                dist[w] = dist[v] + 1
                cola.append(w)
    return dist


adj = arbol_aleatorio(12, 8)
S = suma_distancias(adj)
print("Aristas:", aristas(adj))
print("Sumas:  ", S)
print(f"Mejor sitio para el almacén: nodo {S.index(min(S))} (suma {min(S)})")
for semilla in range(80):
    adj = arbol_aleatorio(random.Random(semilla).randint(1, 40), semilla)
    assert suma_distancias(adj) == [sum(bfs(adj, v)) for v in range(len(adj))]

Aristas: [(0, 1), (0, 7), (1, 2), (1, 3), (1, 4), (1, 5), (1, 8), (2, 9), (3, 10), (5, 6), (8, 11)]
Sumas:   [24, 16, 24, 24, 26, 24, 34, 34, 24, 34, 34, 34]
Mejor sitio para el almacén: nodo 1 (suma 16)


## 7. Excentricidad de cada nodo y centro del árbol

La excentricidad de $v$ es la distancia al nodo más lejano. Calcularla para todos con *rerooting*: $A[v]$ es la mayor profundidad bajando y $B[v]$ la mayor distancia saliendo por el padre, $B[h] = 1 + \max\big(B[p],\ \text{mejor } A[h'] + 1 \text{ entre los hermanos } h' \neq h\big)$, para lo que basta guardar las dos mejores ramas de cada nodo. El centro (uno o dos nodos) es el de menor excentricidad. Comprobarlo con BFS y verificar que el radio es $\lceil d/2 \rceil$, siendo $d$ el diámetro.

In [7]:
def excentricidades(adj):
    n = len(adj)
    padre, orden = colgar(adj)
    A = [0] * n
    mejores = [(0, -1, 0)] * n  # (mejor rama, hijo que la da, segunda rama)
    for v in reversed(orden):
        m1, quien, m2 = 0, -1, 0
        for h in adj[v]:
            if h != padre[v]:
                d = A[h] + 1
                if d > m1:
                    m1, quien, m2 = d, h, m1
                elif d > m2:
                    m2 = d
        A[v], mejores[v] = m1, (m1, quien, m2)
    B = [0] * n
    for v in orden[1:]:
        p = padre[v]
        m1, quien, m2 = mejores[p]
        hermano = m2 if quien == v else m1
        B[v] = 1 + max(B[p], hermano)
    return [max(a, b) for a, b in zip(A, B)]


adj = arbol_aleatorio(12, 8)
E = excentricidades(adj)
radio = min(E)
print("Excentricidades:", E)
print("Centro:", [v for v in range(len(adj)) if E[v] == radio], "radio:", radio, "diámetro:", max(E))
for semilla in range(80):
    adj = arbol_aleatorio(random.Random(semilla).randint(1, 40), semilla)
    E = excentricidades(adj)
    assert E == [max(bfs(adj, v)) for v in range(len(adj))]
    assert min(E) == (max(E) + 1) // 2
    assert 1 <= E.count(min(E)) <= 2

Excentricidades: [3, 2, 3, 3, 3, 3, 4, 4, 3, 4, 4, 4]
Centro: [1] radio: 2 diámetro: 4


## 8. Contar los subárboles conexos

Contar cuántos conjuntos de nodos no vacíos forman un trozo conexo del árbol. Si $K[v]$ cuenta los que tienen a $v$ como nodo más alto, cada hijo se deja fuera o aporta uno de sus $K[h]$ trozos: $K[v] = \prod_h (1 + K[h])$, y la respuesta es $\sum_v K[v]$. Comprobarlo contra todos los subconjuntos y comprobar que en un camino de $n$ nodos sale $n(n+1)/2$ y en una estrella $2^{n-1} + n - 1$.

In [8]:
def subarboles_conexos(adj):
    padre, orden = colgar(adj)
    K = [1] * len(adj)
    for v in reversed(orden):
        if padre[v] >= 0:
            K[padre[v]] *= 1 + K[v]
    return sum(K)


def es_conexo(adj, nodos):
    nodos = set(nodos)
    inicio = next(iter(nodos))
    vistos, pila = {inicio}, [inicio]
    while pila:
        v = pila.pop()
        for w in adj[v]:
            if w in nodos and w not in vistos:
                vistos.add(w)
                pila.append(w)
    return vistos == nodos


def subarboles_bruto(adj):
    n = len(adj)
    return sum(es_conexo(adj, c) for k in range(1, n + 1) for c in combinations(range(n), k))


def camino(n):
    return [[w for w in (v - 1, v + 1) if 0 <= w < n] for v in range(n)]


def estrella(n):
    return [list(range(1, n))] + [[0] for _ in range(n - 1)]


print("Árbol de 12 nodos:", subarboles_conexos(arbol_aleatorio(12, 8)))
for n in range(1, 13):
    assert subarboles_conexos(camino(n)) == n * (n + 1) // 2
    assert subarboles_conexos(estrella(n)) == 2 ** (n - 1) + n - 1
for semilla in range(40):
    adj = arbol_aleatorio(random.Random(semilla).randint(1, 12), semilla)
    assert subarboles_conexos(adj) == subarboles_bruto(adj)

Árbol de 12 nodos: 502


## 9. Conjunto dominante mínimo: vigilancia de la costa

En una red de torres de vigilancia con forma de árbol, una torre con vigía controla la suya y las vecinas. Calcular el mínimo de vigías para que todas las torres estén controladas. Hacen falta tres estados por nodo: $V$ (tiene vigía), $C$ (sin vigía pero cubierto por algún hijo) y $L$ (sin vigía ni hijo con vigía, así que lo cubrirá su padre):

$$V[v] = 1 + \sum_h \min(V, C, L)[h], \qquad L[v] = \sum_h C[h], \qquad C[v] = \sum_h \min(V, C)[h] + \text{recargo}$$

donde el recargo obliga a que al menos un hijo esté en $V$ (es la menor diferencia $V[h] - \min(V[h], C[h])$). Comprobarlo contra fuerza bruta.

In [9]:
INF = float("inf")


def dominante_minimo(adj, raiz=0):
    n = len(adj)
    padre, orden = colgar(adj, raiz)
    V, C, L = [1] * n, [0] * n, [0] * n
    recargo = [INF] * n
    for v in reversed(orden):
        if recargo[v] != INF:
            C[v] += recargo[v]
        else:
            C[v] = INF  # una hoja no puede estar cubierta por un hijo
        p = padre[v]
        if p >= 0:
            V[p] += min(V[v], C[v], L[v])
            L[p] += C[v]
            C[p] += min(V[v], C[v])
            recargo[p] = min(recargo[p], V[v] - min(V[v], C[v]))
    return min(V[raiz], C[raiz])


def dominante_bruto(adj):
    n = len(adj)
    for k in range(1, n + 1):
        for vigias in combinations(range(n), k):
            s = set(vigias)
            if all(v in s or any(w in s for w in adj[v]) for v in range(n)):
                return k


adj = arbol_aleatorio(14, 11)
print("Aristas:", aristas(adj))
print("Vigías necesarios:", dominante_minimo(adj))
print("Camino de 10 torres:", dominante_minimo(camino(10)), " estrella de 10:", dominante_minimo(estrella(10)))
assert dominante_minimo(camino(10)) == 4 and dominante_minimo(estrella(10)) == 1
for semilla in range(60):
    adj = arbol_aleatorio(random.Random(semilla).randint(1, 12), semilla)
    assert dominante_minimo(adj) == dominante_bruto(adj)

Aristas: [(0, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 9), (1, 13), (2, 8), (2, 12), (3, 7), (4, 6), (4, 11), (7, 10)]
Vigías necesarios: 4
Camino de 10 torres: 4  estrella de 10: 1


## 10. Emparejamiento máximo en un árbol

En un campamento, los chicos que son vecinos en el árbol de amistades pueden compartir tienda de dos. Calcular cuántas parejas se pueden formar como máximo, sin repetir a nadie. Con $L[v]$ el mejor del subárbol dejando $v$ libre y $E[v]$ el mejor emparejando $v$ con uno de sus hijos:

$$L[v] = \sum_h \max(L[h], E[h]), \qquad E[v] = \max_h \big(L[v] - \max(L[h], E[h]) + L[h] + 1\big)$$

Comprobarlo contra una búsqueda exhaustiva de conjuntos de aristas disjuntas y contra el voraz clásico de los árboles (emparejar cada hoja libre con su padre, de abajo arriba).

In [10]:
def emparejamiento_maximo(adj):
    padre, orden = colgar(adj)
    n = len(adj)
    L, E = [0] * n, [-INF] * n
    for v in reversed(orden):
        hijos = [h for h in adj[v] if h != padre[v]]
        L[v] = sum(max(L[h], E[h]) for h in hijos)
        for h in hijos:
            E[v] = max(E[v], L[v] - max(L[h], E[h]) + L[h] + 1)
    raiz = orden[0]
    return max(L[raiz], E[raiz])


def emparejamiento_voraz(adj):
    padre, orden = colgar(adj)
    libre = [True] * len(adj)
    parejas = 0
    for v in reversed(orden):
        p = padre[v]
        if p >= 0 and libre[v] and libre[p]:
            libre[v] = libre[p] = False
            parejas += 1
    return parejas


def emparejamiento_bruto(adj):
    lista = aristas(adj)
    for k in range(len(adj) // 2, -1, -1):
        for elegidas in combinations(lista, k):
            extremos = [x for arista in elegidas for x in arista]
            if len(extremos) == len(set(extremos)):
                return k


adj = arbol_aleatorio(13, 12)
print("Aristas:", aristas(adj))
print("Parejas:", emparejamiento_maximo(adj))
for semilla in range(60):
    adj = arbol_aleatorio(random.Random(semilla).randint(1, 13), semilla)
    assert emparejamiento_maximo(adj) == emparejamiento_voraz(adj) == emparejamiento_bruto(adj)
# en un árbol, emparejamiento máximo = cubrimiento mínimo (teorema de König)
for semilla in range(60):
    adj = arbol_aleatorio(random.Random(semilla).randint(1, 60), semilla)
    assert emparejamiento_maximo(adj) == cubrimiento_minimo(adj)

Aristas: [(0, 1), (0, 7), (1, 2), (1, 5), (2, 3), (2, 4), (3, 6), (4, 10), (5, 8), (7, 9), (7, 12), (10, 11)]
Parejas: 6


## 11. Árboles grandes sin recursión

Comprobar que las funciones aguantan un camino de 200 000 nodos (una versión recursiva desbordaría la pila) y un árbol aleatorio del mismo tamaño, y que los resultados cuadran con fórmulas conocidas: en un camino de $n$ nodos, el diámetro es $n - 1$, la suma de distancias desde un extremo es $n(n-1)/2$, el centro tiene excentricidad $\lfloor n/2 \rfloor$ y el mínimo cubrimiento es $\lfloor n/2 \rfloor$.

In [11]:
n = 200_000
c = camino(n)
c_w = [[(w, 1) for w in vecinos] for vecinos in c]
S = suma_distancias(c)
E = excentricidades(c)
print(f"Camino de {n} nodos: diámetro {diametro(c_w)}, S[0] = {S[0]}, radio {min(E)}, cubrimiento {cubrimiento_minimo(c)}")
assert diametro(c_w) == n - 1 and S[0] == n * (n - 1) // 2
assert min(E) == n // 2 and cubrimiento_minimo(c) == emparejamiento_maximo(c) == n // 2

grande = arbol_aleatorio(n, 2026)
E = excentricidades(grande)
grande_w = [[(w, 1) for w in vecinos] for vecinos in grande]
print(f"Árbol aleatorio: diámetro {max(E)}, radio {min(E)}, cubrimiento {cubrimiento_minimo(grande)}, dominante {dominante_minimo(grande)}")
assert max(E) == diametro(grande_w) == diametro_dos_busquedas(grande_w)
assert cubrimiento_minimo(grande) == emparejamiento_voraz(grande)

Camino de 200000 nodos: diámetro 199999, S[0] = 19999900000, radio 100000, cubrimiento 100000


Árbol aleatorio: diámetro 56, radio 28, cubrimiento 80833, dominante 74882
